In [ ]:
!pip install transformers
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")

output = generator("Hello, how are you?", max_length=50)
print(output)

In [ ]:
from google.colab import files

uploaded = files.upload()
file_path = list(uploaded.keys())[0]

# ✅ 2. Generate Embeddings (Sentence-BERT)
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = model.encode(file_path)
doc_embeddings = np.array(doc_embeddings).astype("float32")

print(type(doc_embeddings))
print(doc_embeddings.shape)

# ✅ Fix (just reshape it)
import numpy as np
import faiss

doc_embeddings = np.array(doc_embeddings)

# reshape single vector into 2D
doc_embeddings = doc_embeddings.reshape(1, -1)

# ✅ 3. Store in Vector DB (FAISS)
!pip install faiss-cpu
import faiss

dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)


# ✅ 4. Retrieval Function

def retrieve(query, top_k=3):
    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = [file_path[i] for i in indices[0]]
    return results

# ✅ 5. Add Text Generation Model
    !pip install transformers
    from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")

# ✅ 6. RAG Pipeline
def rag_pipeline(query):
    retrieved_docs = retrieve(query, top_k=3)

    context = " ".join(retrieved_docs)

    prompt = f"""
    Answer the question based on the context below:

    Context: {context}

    Question: {query}
    Answer:
    """

    response = generator(prompt, max_length=150, num_return_sequences=1)
    answer = response[0]["generated_text"]

    return retrieved_docs, answer

# ✅ 7.Checking output
while True:
  q = input("\nAsk (type exit to stop): ")
  if q.lower() == "exit":
        break
  answer = rag_pipeline(q)
  print("\nAnswer:\n", answer)